# FORGE-MA: Train LLMs to Investigate Misinformation via GRPO

This notebook trains a Qwen2.5-3B-Instruct model on the FORGE-MA adversarial misinformation forensics environment using GRPO.

**Runtime**: T4 GPU (free tier) â€” ~45 minutes

**Requirements**: HuggingFace account + FORGE-MA Space running at `https://YOUR-USERNAME-forge-ma.hf.space`

In [ ]:
!git clone https://huggingface.co/spaces/NeuralHU/forge-rl
!pip install -q ./forge-rl

In [ ]:
# Cell 2 â€” Load model with Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",  # 3B fits free T4
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded.")

In [ ]:
# Cell 3 â€” Environment reward function (rollout_func pattern)
from forge_ma import ForgeEnv, ForgeAction
from trl.experimental.openenv import generate_rollout_completions
from trl import GRPOTrainer

SPACE_URL = "https://YOUR-USERNAME-forge-ma.hf.space"

def _parse_verdict(text: str) -> int:
    t = text.lower()
    if any(w in t for w in ["misinfo", "false", "fabricat", "manipulat", "mislead"]):
        return 10
    elif any(w in t for w in ["satire", "parody", "joke", "humor"]):
        return 11
    elif any(w in t for w in ["verified", "true", "legitimate", "accurate", "real"]):
        return 12
    return 10  # default

def forge_rollout(prompts: list, trainer: GRPOTrainer):
    # Step 1: Generate completions
    outputs = generate_rollout_completions(trainer, prompts)
    completions = [
        tokenizer.decode(o["completion_ids"], skip_special_tokens=True)
        for o in outputs
    ]

    # Step 2: Run each completion through environment
    env_rewards = []
    for completion in completions:
        try:
            with ForgeEnv(base_url=SPACE_URL).sync() as env:
                env.reset()
                # 3 investigation steps
                env.step(ForgeAction(action=0))   # query_source
                env.step(ForgeAction(action=3))   # temporal_audit
                env.step(ForgeAction(action=1))   # cross_reference
                # Submit verdict based on LLM completion
                verdict_action = _parse_verdict(completion)
                result = env.step(ForgeAction(action=verdict_action))
                env_rewards.append(float(result.reward))
        except Exception as e:
            print(f"Env error: {e}")
            env_rewards.append(0.001)

    return {
        "prompt_ids":      [o["prompt_ids"]      for o in outputs],
        "completion_ids":  [o["completion_ids"]  for o in outputs],
        "logprobs":        [o["logprobs"]         for o in outputs],
        "env_reward":      env_rewards,
    }

def reward_fn(completions, **kwargs):
    return [float(r) for r in kwargs.get("env_reward", [0.001] * len(completions))]

In [ ]:
# Cell 4 â€” Dataset
from datasets import Dataset

INVESTIGATION_PROMPT = """You are a misinformation forensics agent. 
You have just investigated a claim using source queries, temporal analysis, 
and cross-referencing. Based on your investigation, submit your verdict.

Respond with one of:
- "This is MISINFO because [reason]"  
- "This is SATIRE because [reason]"
- "This is VERIFIED because [reason]"

Your verdict:"""

dataset = Dataset.from_dict({
    "prompt": [INVESTIGATION_PROMPT] * 256
})
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Cell 5 â€” Baseline evaluation BEFORE training
import numpy as np

def evaluate_agent(n_episodes=20):
    rewards = []
    correct = 0
    for _ in range(n_episodes):
        try:
            with ForgeEnv(base_url=SPACE_URL).sync() as env:
                obs = env.reset()
                env.step(ForgeAction(action=0))
                result = env.step(ForgeAction(action=10))  # always predict misinfo
                rewards.append(float(result.reward))
                if result.reward > 0.5:
                    correct += 1
        except:
            rewards.append(0.001)
    return np.mean(rewards), correct / n_episodes

baseline_reward, baseline_acc = evaluate_agent()
print(f"BASELINE â€” Mean reward: {baseline_reward:.4f} | Accuracy: {baseline_acc:.2%}")

In [ ]:
# Cell 6 â€” Train
from trl import GRPOConfig

config = GRPOConfig(
    output_dir="./forge-grpo",
    use_vllm=True,
    vllm_mode="colocate",
    num_train_epochs=3,
    num_generations=4,
    max_completion_length=256,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    save_steps=50,
    logging_steps=5,
    report_to="none",
    warmup_ratio=0.1,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=config,
    train_dataset=dataset,
    tokenizer=tokenizer,
    rollout_func=forge_rollout,
)

print("Starting training...")
trainer.train()
print("Training complete.")

In [ ]:
# Cell 7 â€” Post-training evaluation + PLOT (COMMIT THIS)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import os

os.makedirs("results", exist_ok=True)

# Extract reward curve from trainer logs
log_history = trainer.state.log_history
steps   = [l["step"]        for l in log_history if "reward_mean" in l]
rewards = [l["reward_mean"] for l in log_history if "reward_mean" in l]

# Post-training evaluation
trained_reward, trained_acc = evaluate_agent()
print(f"TRAINED  â€” Mean reward: {trained_reward:.4f} | Accuracy: {trained_acc:.2%}")
print(f"IMPROVEMENT: {trained_reward - baseline_reward:+.4f} reward | {trained_acc - baseline_acc:+.2%} accuracy")

# Plot 1: Reward curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps, rewards, 'b-', linewidth=2.5, label='Training reward')
ax.axhline(y=baseline_reward, color='r', linestyle='--',
           linewidth=2, label=f'Random baseline ({baseline_reward:.3f})')
ax.set_xlabel("Training Step", fontsize=13)
ax.set_ylabel("Mean Episode Reward", fontsize=13)
ax.set_title("FORGE-MA: LLM Agent Learning Misinformation Detection via GRPO",
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("results/reward_curve.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved: results/reward_curve.png")

# Plot 2: Before/After bar chart
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    ['Random Baseline', 'GRPO Trained'],
    [baseline_reward, trained_reward],
    color=['#e74c3c', '#2ecc71'],
    edgecolor='black', linewidth=1.2,
    width=0.5
)
ax.set_ylabel("Mean Episode Reward", fontsize=13)
ax.set_title("Before vs After GRPO Training\n(FORGE-MA Misinformation Agent)", fontsize=13)
ax.set_ylim(0, 1.0)
for bar, val in zip(bars, [baseline_reward, trained_reward]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("results/before_after.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved: results/before_after.png")
print("\nCOMMIT BOTH PNG FILES TO REPO BEFORE SUBMITTING")